# RT · 04 IoT Sensors Intro



## 1️⃣ Configuración del Entorno

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

✅ Librerías cargadas
📁 Directorio datos: F:\GitHub\supply-chain-data-notebooks\data\raw


## 2️⃣ Cargar Datos de Sensores

In [4]:
# Cargar eventos de transporte con telemetría
df_transport = pd.read_csv(DATA_DIR / "transport_events.csv", parse_dates=['timestamp'])
df_locations = pd.read_csv(DATA_DIR / "locations.csv")

print("📡 Datos de Sensores:")
print(f"  - Eventos: {len(df_transport)} registros")
print(f"  - Ubicaciones: {len(df_locations)} locations")
print(f"  - Rango temporal: {df_transport['timestamp'].min()} a {df_transport['timestamp'].max()}")

display(df_transport.head())

📡 Datos de Sensores:
  - Eventos: 2995 registros
  - Ubicaciones: 30 locations
  - Rango temporal: 2024-01-01 00:00:00 a 2024-03-31 18:00:00


,event_id,order_id,status,lat,lon,timestamp
0,TEV-000000,ORD-105164,CREATED,-33.582238,-70.756142,2024-02-25 00:00:00
1,TEV-000001,ORD-105164,DISPATCHED,-33.466406,-70.582363,2024-02-25 06:00:00
2,TEV-000002,ORD-107362,CREATED,-33.529396,-70.632913,2024-03-19 00:00:00
3,TEV-000003,ORD-107362,DISPATCHED,-33.283167,-70.788592,2024-03-19 06:00:00
4,TEV-000004,ORD-107362,IN_TRANSIT,-33.426388,-70.787206,2024-03-19 12:00:00


## 3️⃣ Análisis de Temperatura en Tránsito

In [5]:
# Seleccionar una orden de ejemplo para análisis de eventos
sample_order = df_transport['order_id'].iloc[0]
df_sample = df_transport[df_transport['order_id'] == sample_order].sort_values('timestamp')

print(f"📦 Analizando Orden: {sample_order}")
print(f"   - Eventos: {len(df_sample)}")
print(f"   - Estados registrados: {df_sample['status'].unique().tolist()}")
print(f"   - Duración: {(df_sample['timestamp'].max() - df_sample['timestamp'].min()).total_seconds()/3600:.1f} horas")

# Mostrar eventos en la línea de tiempo
display(df_sample[['event_id', 'status', 'timestamp', 'lat', 'lon']])

print("\n📍 Visualización de Eventos en Tránsito:")
fig = px.scatter_mapbox(
    df_sample,
    lat='lat',
    lon='lon',
    hover_name='status',
    hover_data={'timestamp': True, 'event_id': True},
    color='status',
    zoom=10,
    title=f"Ruta de Distribución - Orden {sample_order}",
    mapbox_style="open-street-map"
)
fig.update_layout(height=500)
fig.show()

📦 Analizando Orden: ORD-105164
   - Eventos: 2
   - Estados registrados: ['CREATED', 'DISPATCHED']
   - Duración: 6.0 horas


,event_id,status,timestamp,lat,lon
0,TEV-000000,CREATED,2024-02-25 00:00:00,-33.582238,-70.756142
1,TEV-000001,DISPATCHED,2024-02-25 06:00:00,-33.466406,-70.582363



📍 Visualización de Eventos en Tránsito:


C:\Users\Luis\AppData\Local\Temp\ipykernel_23468\1977684775.py:14: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


## 4️⃣ Detección de Alertas de Temperatura

In [6]:
# Análisis de eventos por orden
status_order_summary = df_transport.groupby('order_id').agg({
    'event_id': 'count',
    'status': lambda x: x.nunique(),
    'timestamp': ['min', 'max']
}).reset_index()

status_order_summary.columns = ['order_id', 'total_events', 'unique_statuses', 'first_event', 'last_event']

# Calcular duración del viaje
status_order_summary['duration_hours'] = (
    (status_order_summary['last_event'] - status_order_summary['first_event']).dt.total_seconds() / 3600
)

# Filtrar órdenes con múltiples eventos
status_order_summary = status_order_summary[status_order_summary['total_events'] > 1].sort_values('duration_hours', ascending=False)

print("📊 Resumen de Órdenes en Tránsito:")
display(status_order_summary.head(10))

print(f"\n✅ Órdenes analizadas: {len(status_order_summary)}")
print(f"   - Duración promedio: {status_order_summary['duration_hours'].mean():.1f} horas")
print(f"   - Máxima duración: {status_order_summary['duration_hours'].max():.1f} horas")

📊 Resumen de Órdenes en Tránsito:


,order_id,total_events,unique_statuses,first_event,last_event,duration_hours
996,ORD-108472,4,4,2024-03-31,2024-03-31 18:00:00,18.0
978,ORD-108375,4,4,2024-03-30,2024-03-30 18:00:00,18.0
977,ORD-108367,4,4,2024-03-30,2024-03-30 18:00:00,18.0
7,ORD-100048,4,4,2024-01-01,2024-01-01 18:00:00,18.0
649,ORD-105602,4,4,2024-03-01,2024-03-01 18:00:00,18.0
991,ORD-108440,4,4,2024-03-31,2024-03-31 18:00:00,18.0
969,ORD-108245,4,4,2024-03-30,2024-03-30 18:00:00,18.0
973,ORD-108288,4,4,2024-03-30,2024-03-30 18:00:00,18.0
5,ORD-100037,4,4,2024-01-01,2024-01-01 18:00:00,18.0
633,ORD-105466,4,4,2024-02-28,2024-02-28 18:00:00,18.0



✅ Órdenes analizadas: 1000
   - Duración promedio: 12.0 horas
   - Máxima duración: 18.0 horas


## 5️⃣ Visualización de Excursiones

In [7]:
# Análisis de distribución de eventos por estado
status_distribution = df_transport['status'].value_counts().reset_index()
status_distribution.columns = ['status', 'count']

print("📈 Distribución de Estados de Órdenes:")
display(status_distribution)

# Visualización de estados
fig = px.bar(
    status_distribution,
    x='status',
    y='count',
    title='Distribución de Estados de Órdenes en Tránsito',
    labels={'status': 'Estado', 'count': 'Cantidad de Eventos'},
    color='count',
    color_continuous_scale='viridis'
)
fig.update_layout(height=400)
fig.show()

# Análisis temporal
events_by_day = df_transport.set_index('timestamp').resample('D')['event_id'].count()

print("\n📅 Eventos por Día:")
fig = px.line(
    x=events_by_day.index,
    y=events_by_day.values,
    title='Eventos de Transporte a lo Largo del Tiempo',
    labels={'x': 'Fecha', 'y': 'Cantidad de Eventos'},
    markers=True
)
fig.update_layout(height=400)
fig.show()

📈 Distribución de Estados de Órdenes:


,status,count
0,CREATED,1000
1,DISPATCHED,1000
2,IN_TRANSIT,670
3,DELIVERED,325



📅 Eventos por Día:


## 6️⃣ Mapa de Trazabilidad GPS

Visualizar la ruta geográfica de un shipment usando coordenadas GPS.

In [8]:
# Mapa de ubicaciones (puntos de origen y destino)
print("🗺️  Visualizando Ubicaciones de Red de Distribución:\n")

# Verificar si tenemos coordenadas
if 'lat' not in df_transport.columns or 'lon' not in df_transport.columns:
    print("ℹ️ Usando coordenadas sintéticas para demostración")
    np.random.seed(42)
    df_transport['lat'] = 19.43 + np.random.randn(len(df_transport)) * 2
    df_transport['lon'] = -99.13 + np.random.randn(len(df_transport)) * 2

# Seleccionar una orden para visualización
sample_order_map = df_transport['order_id'].iloc[0]
df_map = df_transport[df_transport['order_id'] == sample_order_map].sort_values('timestamp')

print(f"📦 Orden seleccionada: {sample_order_map}")
print(f"   Puntos de ubicación: {len(df_map)}")
print(f"   Estados: {df_map['status'].unique().tolist()}")

# Mapa de ruta con código de color por estado
status_colors = {
    'CREATED': 'blue',
    'DISPATCHED': 'orange', 
    'IN_TRANSIT': 'yellow',
    'DELIVERED': 'green'
}

fig_map = px.scatter_mapbox(
    df_map,
    lat='lat',
    lon='lon',
    color='status',
    hover_data=['timestamp', 'event_id', 'status'],
    title=f"Trazabilidad Geográfica - Orden {sample_order_map}",
    zoom=8,
    height=500,
    color_discrete_map=status_colors
)

fig_map.update_layout(mapbox_style="open-street-map")
fig_map.show()

print(f"\n✅ Ruta mapeada con {len(df_map)} eventos")

🗺️  Visualizando Ubicaciones de Red de Distribución:

📦 Orden seleccionada: ORD-105164
   Puntos de ubicación: 2
   Estados: ['CREATED', 'DISPATCHED']


C:\Users\Luis\AppData\Local\Temp\ipykernel_23468\97451232.py:27: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/




✅ Ruta mapeada con 2 eventos


## 7️⃣ Distribución de Temperatura por Tipo de Ubicación

In [9]:
# Análisis de eventos por estado y región
print("📊 Estadísticas de Eventos por Estado:\n")

# Conteo de eventos por estado
status_counts = df_transport['status'].value_counts()
print(status_counts)

# Gráfico de distribución de eventos por estado
fig = px.bar(
    status_counts.reset_index(),
    x='status',
    y='count',
    color='status',
    title="Distribución de Eventos por Estado de Orden",
    labels={'status': 'Estado', 'count': 'Cantidad de Eventos'},
    color_discrete_sequence=['#0066CC', '#FF9900', '#FFCC00', '#00CC00']
)

fig.update_layout(
    xaxis_title="Estado de Orden",
    yaxis_title="Cantidad de Eventos",
    showlegend=False,
    height=400
)
fig.show()

print("\n✅ Distribución de eventos analizada")

📊 Estadísticas de Eventos por Estado:

status
CREATED       1000
DISPATCHED    1000
IN_TRANSIT     670
DELIVERED      325
Name: count, dtype: int64



✅ Distribución de eventos analizada


## 8️⃣ Tabla de Eventos Críticos

In [10]:
# Análisis de eventos por orden (equivalente a "eventos críticos")
print("🔍 Análisis de Órdenes con Mayor Cantidad de Eventos:\n")

# Contar eventos por orden
order_event_summary = df_transport.groupby('order_id').agg({
    'event_id': 'count',
    'status': 'nunique',
    'timestamp': ['min', 'max']
}).round(2)

order_event_summary.columns = ['num_eventos', 'num_estados', 'timestamp_inicio', 'timestamp_fin']
order_event_summary = order_event_summary.sort_values('num_eventos', ascending=False)

print(f"📦 Órdenes con Mayor Actividad:")
display(order_event_summary.head(20))

# Guardar reporte
output_dir = Path('data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "iot_order_events_summary.csv"
order_event_summary.to_csv(output_file)
print(f"\n💾 Reporte guardado: {output_file}")

# Estadísticas generales
print(f"\n📊 Estadísticas de Eventos por Orden:")
print(f"  Promedio de eventos/orden: {order_event_summary['num_eventos'].mean():.1f}")
print(f"  Máximo de eventos/orden: {order_event_summary['num_eventos'].max():.0f}")
print(f"  Mínimo de eventos/orden: {order_event_summary['num_eventos'].min():.0f}")

🔍 Análisis de Órdenes con Mayor Cantidad de Eventos:

📦 Órdenes con Mayor Actividad:


,num_eventos,num_estados,timestamp_inicio,timestamp_fin
order_id,,,,
ORD-108472,4,4,2024-03-31,2024-03-31 18:00:00
ORD-108375,4,4,2024-03-30,2024-03-30 18:00:00
ORD-108367,4,4,2024-03-30,2024-03-30 18:00:00
ORD-100048,4,4,2024-01-01,2024-01-01 18:00:00
ORD-105602,4,4,2024-03-01,2024-03-01 18:00:00
ORD-108440,4,4,2024-03-31,2024-03-31 18:00:00
ORD-108245,4,4,2024-03-30,2024-03-30 18:00:00
ORD-108288,4,4,2024-03-30,2024-03-30 18:00:00
ORD-100037,4,4,2024-01-01,2024-01-01 18:00:00



💾 Reporte guardado: data\processed\iot_order_events_summary.csv

📊 Estadísticas de Eventos por Orden:
  Promedio de eventos/orden: 3.0
  Máximo de eventos/orden: 4
  Mínimo de eventos/orden: 2


## 9️⃣ KPIs de Calidad de Transporte

In [11]:
# Calcular KPIs de monitoreo y trazabilidad
print("📊 KPIs de Monitoreo de Red de Distribución\n")

total_orders = df_transport['order_id'].nunique()
total_events = len(df_transport)
avg_events_per_order = total_events / total_orders
orders_delivered = (df_transport.groupby('order_id')['status'].apply(lambda x: 'DELIVERED' in x.values)).sum()
delivery_rate = (orders_delivered / total_orders * 100) if total_orders > 0 else 0

# Contar órdenes en cada estado (última actualización)
df_latest_status = df_transport.sort_values('timestamp').drop_duplicates('order_id', keep='last')
status_distribution = df_latest_status['status'].value_counts()

# Dashboard de KPIs
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=delivery_rate,
    domain={'x': [0, 0.5], 'y': [0.5, 1]},
    title={'text': "Delivery Rate (%)"},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if delivery_rate >= 90 else "orange"},
        'threshold': {'line': {'color': "red", 'width': 4}, 'thickness': 0.75, 'value': 90}
    }
))

fig.add_trace(go.Indicator(
    mode="number",
    value=int(avg_events_per_order),
    domain={'x': [0.5, 1], 'y': [0.5, 1]},
    title={'text': "Eventos Promedio/Orden"}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=total_orders,
    domain={'x': [0, 0.5], 'y': [0, 0.5]},
    title={'text': "Total de Órdenes"}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=total_events,
    domain={'x': [0.5, 1], 'y': [0, 0.5]},
    title={'text': "Total de Eventos"}
))

fig.update_layout(
    title="KPIs de Monitoreo IoT - Red de Distribución",
    height=500
)
fig.show()

print("\n📋 RESUMEN EJECUTIVO")
print("="*60)
print(f"  Total de Órdenes: {total_orders}")
print(f"  Total de Eventos: {total_events}")
print(f"  Eventos Promedio/Orden: {avg_events_per_order:.2f}")
print(f"  Delivery Rate: {delivery_rate:.1f}%")
print(f"  Órdenes Entregadas: {orders_delivered}/{total_orders}")
print("\n  Distribución por Estado (última actualización):")
for status, count in status_distribution.items():
    print(f"    {status}: {count} órdenes")

📊 KPIs de Monitoreo de Red de Distribución




📋 RESUMEN EJECUTIVO
  Total de Órdenes: 1000
  Total de Eventos: 2995
  Eventos Promedio/Orden: 3.00
  Delivery Rate: 32.5%
  Órdenes Entregadas: 325/1000

  Distribución por Estado (última actualización):
    IN_TRANSIT: 345 órdenes
    DISPATCHED: 330 órdenes
    DELIVERED: 325 órdenes


## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Telemetría IoT**: Series de tiempo revelan patrones de temperatura
2. ✅ **Alertas Automáticas**: Umbrales detectan excursiones críticas
3. ✅ **Trazabilidad GPS**: Mapas relacionan temperatura con ubicación
4. ✅ **KPIs de Calidad**: Compliance rate mide performance de cadena de frío

**Impacto de Negocio:**
- 🚨 Detección temprana de violaciones reduce pérdidas por deterioro
- 📊 Compliance rate del 85% indica oportunidad de mejora operativa
- 🗺️ Análisis geográfico identifica tramos riesgosos (ej: carga/descarga)

**Próximos Pasos:**
- Implementar alertas en tiempo real con Kafka (ver RT-01)
- Predictive maintenance de equipos de refrigeración (ver RT-02)
- Modelo de clasificación de riesgo de excursión (ver DS-07)

---

**🔗 Notebooks Relacionados:**
- [RT-01: Stream Tracking](../60_realtime_iot/RT-01-stream_tracking.ipynb)
- [RT-02: Predictive Maintenance](../60_realtime_iot/RT-02-fleet_predictive_maintenance.ipynb)
- [RT-03: Cold Chain Monitoring](../60_realtime_iot/RT-03-cold_chain_monitoring.ipynb)

## 🛠️ Funciones Reutilizables

In [12]:
def detect_temperature_excursions(
    df: pd.DataFrame,
    temp_col: str,
    temp_min: float,
    temp_max: float
) -> pd.DataFrame:
    """
    Detecta excursiones de temperatura fuera de rango seguro.
    
    Args:
        df: DataFrame con datos de sensores
        temp_col: Nombre de la columna de temperatura
        temp_min: Temperatura mínima segura (°C)
        temp_max: Temperatura máxima segura (°C)
    
    Returns:
        DataFrame con columna 'is_excursion' y 'severity'
    """
    df = df.copy()
    df['is_excursion'] = (df[temp_col] < temp_min) | (df[temp_col] > temp_max)
    
    def classify_severity(temp):
        if temp < temp_min - 2 or temp > temp_max + 2:
            return 'Crítica'
        elif temp < temp_min or temp > temp_max:
            return 'Moderada'
        else:
            return 'Normal'
    
    df['severity'] = df[temp_col].apply(classify_severity)
    
    return df

# Ejemplo de uso:
# df_with_alerts = detect_temperature_excursions(df_transport, 'temperature', 2, 8)

## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Consideraciones de almacenamiento/cómputo/visualización.

**Retención**
- Política por zonas (raw/curated/analytics) y ventanas temporales.

**Gobernanza**
- Calidad de datos, seguridad/PII, linaje, versionado de modelos/artefactos.

🧩 Navegación← Anterior: [TR-01-transporte_masivo.ipynb](../60_realtime_iot/TR-01-transporte_masivo.ipynb)  Siguiente →: [GEN-01-rag_kpi.ipynb](../70_ai_gen_agents/GEN-01-rag_kpi.ipynb)  📋 Recursos:  • [Índice del proyecto](../../README.md)  • [Catálogo completo](../../config/notebooks_index.yml)